# 04 — Phase 3: ECG + Metadata Fusion

This notebook trains a late-fusion model using the ECG waveform and **age/sex only**. It does not use SCP codes or diagnosis-derived report text because those can reveal the target label.

Checkpoint selection uses a held-out validation split from site 1. Site 2 is evaluated only after selecting the checkpoint. Height and weight are excluded from this primary experiment because they are missing in 98% of site-2 records.

## 1. Confirm repository and data
Run the setup notebook first (`00_colab_setup.ipynb`) or make sure the current directory is the repository root and `data/raw/ptbxl/ptbxl_database.csv` exists.

In [ ]:
from pathlib import Path
print("Repo root:", Path.cwd())
print("PTB-XL metadata exists:", Path("data/raw/ptbxl/ptbxl_database.csv").exists())
assert Path("data/raw/ptbxl/ptbxl_database.csv").exists(), "Run notebook 00 / download PTB-XL first."

## 2. Train fusion model
This also trains a metadata-only comparison model and evaluates the best source-validation checkpoint on site 2 under full, signal-only, and metadata-only input conditions.

In [ ]:
!python -m src.training.trainer_fusion --config configs/fusion_full.yaml

## 3. Inspect results
Primary metric: HYP F1/precision/recall. Compare the full fusion model against the signal-only condition and metadata-only baseline. A higher overall accuracy alone is not sufficient.

In [ ]:
import json
from pathlib import Path
result_path = Path("experiments/results/fusion_metadata_site1v2/metrics.json")
assert result_path.exists(), f"Missing results: {result_path}"
results = json.loads(result_path.read_text())
print(json.dumps(results, indent=2))

## Interpretation checklist
- Did the fusion model improve HYP F1 compared with the signal-only condition?
- Did metadata alone perform strongly? If yes, fusion gains need cautious interpretation.
- Site 2 was not used for checkpoint selection.
- This is ECG + demographic metadata, not independent physician-report fusion; update the proposal framing if this becomes the primary modality pair.